# FedGRAA — Reproduction Notebook (Local & Google Colab)

Run the **real** CIC-IDS-2017 / UNSW-NB15 experiments end-to-end.

**Two execution paths are supported:**

| Path | When to use |
|------|-------------|
| **Local** (VS Code / JupyterLab) | You already have the CSV files in `data/CIC-IDS-2017/` inside this repo. Use **Step 0** below and skip Steps 1-2. |
| **Google Colab** | You need to upload code and/or data. Follow Steps 1-2. |

The implementation is pure NumPy — no TensorFlow / GPU needed. Full 200 rounds on CIC-IDS-2017 takes ~30–120 min on CPU; use `--rounds 60` for a smoke-test first.

## Step 0 — Local execution (VS Code / JupyterLab)

Skip this section if you are running on **Google Colab** and go straight to Step 1.

Run the cell below once to wire up paths and verify the package is importable.

In [ ]:
"""Local setup — gap 3.10 fix.

Sets REPO_DIR and DATA_DIR from the repository layout so the rest of the
notebook works identically to the Colab path without any uploads.
Run this cell INSTEAD OF the Colab Step-1 / Step-2 cells.
"""
import os, sys, pathlib

# Locate the PAKDD folder that contains fedgraa/__init__.py
_here = pathlib.Path().resolve()
# If running from within PAKDD/, __init__.py is at fedgraa/__init__.py
_candidates = [_here, _here.parent]
REPO_DIR = None
for _c in _candidates:
    if (_c / "fedgraa" / "__init__.py").exists():
        REPO_DIR = str(_c)
        break
assert REPO_DIR, (
    "Could not auto-detect REPO_DIR. Open this notebook from inside the PAKDD/ folder."
)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Point at the real CSV data that ships with the repo.
DATA_DIR_CIC  = os.path.join(REPO_DIR, "data", "CIC-IDS-2017")
DATA_DIR_UNSW = os.path.join(REPO_DIR, "data", "UNSW-NB15")

import fedgraa
from fedgraa.config import CIC_IDS_2017, UNSW_NB15, DATASETS

print("fedgraa version :", fedgraa.__version__)
print("REPO_DIR        :", REPO_DIR)
print("CIC-IDS-2017 CSV dir :", DATA_DIR_CIC,
      f"({'exists' if os.path.isdir(DATA_DIR_CIC) else 'MISSING'})")
print("UNSW-NB15 CSV dir    :", DATA_DIR_UNSW,
      f"({'exists' if os.path.isdir(DATA_DIR_UNSW) else 'MISSING — will use synthetic data'})")
print("\nAvailable datasets:", sorted(DATASETS.keys()))
print(f"CIC-IDS-2017: {CIC_IDS_2017.n_clients} clients, {CIC_IDS_2017.n_classes} classes")
print(f"UNSW-NB15   : {UNSW_NB15.n_clients} clients, {UNSW_NB15.n_classes} classes")

In [ ]:
# Local gamma sweep — Figure 1 reproduction.
# Uses real CSV data if the folder is populated; falls back to synthetic transparently.
from fedgraa.experiment import run_gamma_sweep, summarize_gamma_sweep, tabulate_history, save_results

DATA_DIR = DATA_DIR_CIC  # change to DATA_DIR_UNSW to run UNSW-NB15

results = run_gamma_sweep(
    dataset="CIC-IDS-2017",
    data_dir=DATA_DIR if os.path.isdir(DATA_DIR) else None,
    rounds=200,          # use rounds=60 for a quick smoke-test
    gammas=(0.0, 0.5, 1.0),
    n_rare=3,
    seed=0,
    cache_dir=os.path.join(REPO_DIR, ".cache"),
)

print("\n=== Final-round summary (Figure 1 style) ===")
print(summarize_gamma_sweep(results))

# Save results so they don't need to be re-run.
save_results(results, os.path.join(REPO_DIR, "results_gamma_sweep_cic.json"))
print("\nResults saved to results_gamma_sweep_cic.json")

In [ ]:
# Full per-round accuracy table (gap 3.2 fix) + dual-metric Figure 1 plot.
from fedgraa.plot import plot_dual_metric, plot_weight_evolution, plot_composite_loss

# Per-round table for one method.
print("=== Per-round table for FedGRAA (γ=1) ===")
print(tabulate_history(results["fedgraa_g1.0"].history))

# Figure 1: both all-class and rare-class accuracy side by side.
fig1 = plot_dual_metric(results, title="CIC-IDS-2017 — Figure 1 (γ sweep)")
fig1.savefig(os.path.join(REPO_DIR, "figure1_cic.png"), dpi=150)
print("\nFigure 1 saved to figure1_cic.png")

# Weight evolution plot (simplex-constraint sanity check, gap 1.1).
fig_w = plot_weight_evolution(
    results["fedgraa_g1.0"].history,
    client_labels=CIC_IDS_2017.labels[1:],  # skip Benign
    top_k=5,
    title="CIC-IDS-2017 — FedGRAA weight evolution (γ=1)",
)
fig_w.savefig(os.path.join(REPO_DIR, "weights_cic.png"), dpi=150)

# Composite-loss plot (Lemma 1 check, gap 3.7).
fedgraa_hists = {k: v.history for k, v in results.items() if "fedgraa" in k}
fig_cl = plot_composite_loss(fedgraa_hists, title="CIC-IDS-2017 — Composite loss L'(w^t)")
fig_cl.savefig(os.path.join(REPO_DIR, "composite_loss_cic.png"), dpi=150)
print("Weight and composite-loss plots saved.")

In [ ]:
# n-sweep (Figures 2-3) locally.
from fedgraa.experiment import run_n_sweep, load_results
from fedgraa.plot import plot_n_sweep

n_results = run_n_sweep(
    "CIC-IDS-2017",
    ns=(3, 5, 7),
    gamma=1.0,
    rounds=200,
    data_dir=DATA_DIR if os.path.isdir(DATA_DIR) else None,
    seed=0,
    cache_dir=os.path.join(REPO_DIR, ".cache"),
)

fig23 = plot_n_sweep(n_results, metric="last_n", title="CIC-IDS-2017 — Figures 2-3 (n sweep)")
fig23.savefig(os.path.join(REPO_DIR, "figure2_3_cic.png"), dpi=150)

print("\nn | FedAvg(last-n) | Static(last-n) | FedGRAA(last-n)")
print("--+----------------+----------------+----------------")
for n, res in n_results.items():
    print(
        f"{n} | {res['fedavg'].final_last_n_acc*100:13.2f}  | "
        f"{res['static'].final_last_n_acc*100:13.2f}  | "
        f"{res['fedgraa'].final_last_n_acc*100:13.2f}"
    )
print("\nFigures 2-3 saved to figure2_3_cic.png")

## Step 1 — Get the FedGRAA code into Colab

Pick **one** path by setting `USE_DRIVE_FOR_CODE`:
* **Upload a zip** (default) — on your laptop, zip the `PAKDD` folder (the one containing the `fedgraa/` package) and upload it here.
* **Google Drive** — if the repo already lives in your Drive, set the flag to `True` and edit `REPO_DIR`.

In [ ]:
USE_DRIVE_FOR_CODE = False  # True if your PAKDD repo is already in Google Drive

import os, sys

if USE_DRIVE_FOR_CODE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Research_Project/PAKDD'  # <-- EDIT to your path
else:
    from google.colab import files
    import zipfile, io
    print('Upload a .zip of your PAKDD folder (must contain the fedgraa/ package)...')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
        z.extractall('/content/fedgraa_repo')
    # Locate the directory that actually contains the fedgraa package.
    REPO_DIR = None
    for root, dirs, _ in os.walk('/content/fedgraa_repo'):
        if os.path.exists(os.path.join(root, 'fedgraa', '__init__.py')):
            REPO_DIR = root
            break
    assert REPO_DIR, 'Could not find the fedgraa package inside the uploaded zip.'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

In [ ]:
# Dependencies (Colab already ships numpy / pandas / scikit-learn; this is a no-op safety net).
!pip install -q "numpy>=1.24" "pandas>=2.0" "scikit-learn>=1.3" matplotlib

import fedgraa
from fedgraa.config import CIC_IDS_2017, UNSW_NB15
print('fedgraa version:', fedgraa.__version__)
print('CIC-IDS-2017:', CIC_IDS_2017.n_clients, 'clients,', CIC_IDS_2017.n_classes, 'classes')
print('UNSW-NB15  :', UNSW_NB15.n_clients, 'clients,', UNSW_NB15.n_classes, 'classes')

## Step 2 — Get the real CIC-IDS-2017 CSVs

The loader reads **every `.csv` in a folder**, concatenates them, cleans `inf`/`NaN`, maps the real label spellings onto the paper's classes (including `Web Attack – Brute Force → Brute Force`), and caps frequent classes at 10,000.

Pick **one** source below and set `DATA_DIR` to the folder holding the CSVs.

### Option A — Google Drive (most reliable)
Upload the CIC-IDS-2017 `MachineLearningCVE` CSVs to a Drive folder, then mount and point `DATA_DIR` at it.

### Option B — Kaggle API
Upload your `kaggle.json` token and download a CICIDS-2017 dataset. Replace the slug with the dataset you choose (e.g. one that contains the 8 `MachineLearningCVE` CSVs).

In [ ]:
# === Option A: Google Drive ===
MOUNT_DRIVE_FOR_DATA = True
if MOUNT_DRIVE_FOR_DATA:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('drive already mounted or:', e)

# EDIT this to the folder that holds your CIC-IDS-2017 *.csv files:
DATA_DIR = '/content/drive/MyDrive/datasets/CIC-IDS-2017'

In [ ]:
# === Option B: Kaggle API (uncomment to use instead of Drive) ===
from google.colab import files
print('Upload your kaggle.json API token...')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

# Replace <owner>/<dataset-slug> with a CICIDS-2017 dataset that contains the
# MachineLearningCVE CSVs. Inspect the extracted folder and set DATA_DIR to it.
!kaggle datasets download -d <owner>/<dataset-slug> -p /content/cicids --unzip
DATA_DIR = '/content/cicids'  # adjust to the subfolder that actually holds the .csv files

In [ ]:
# Sanity-check the data folder before the (slow) training run.
import glob, os
import pandas as pd

csvs = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print('DATA_DIR =', DATA_DIR)
print('found', len(csvs), 'CSV files:')
for c in csvs:
    print('  -', os.path.basename(c))
assert csvs, f'No CSV files found in {DATA_DIR}. Fix DATA_DIR above.'

# Peek at the label column distribution of the first file.
head = pd.read_csv(csvs[0], nrows=20000)
head.columns = [c.strip() for c in head.columns]
label_col = next((c for c in head.columns if c.strip().lower() in ('label', 'attack', 'attack_cat')), head.columns[-1])
print('\nlabel column:', label_col)
print(head[label_col].value_counts().head(20))

## Step 3 — Run the real experiment

This is the exact command you asked for:

```
python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir <DATA_DIR> --rounds 200
```

It sweeps γ ∈ {0, 0.5, 1} and reports overall accuracy and the **last-n rare-class accuracy** (the metric the paper improves).

> **Smoke-test first** with 60 rounds (next cell). Once that runs end-to-end, launch the full 200-round cell.

In [ ]:
# Smoke test: 60 rounds, print metrics every 20 rounds.
!python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --rounds 60 --eval-every 20

In [ ]:
# THE REAL DEAL: full 200-round gamma sweep (this is the slow one).
!python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --rounds 200

## Step 4 — Run programmatically and plot

Same sweep, but called from Python so you can capture the numbers and chart them. Use this if you want figures instead of just stdout.

In [ ]:
from fedgraa.experiment import run_gamma_sweep, summarize_gamma_sweep

# Use rounds=200 to match the paper; lower it while iterating.
results = run_gamma_sweep(
    dataset='CIC-IDS-2017',
    data_dir=DATA_DIR,
    rounds=200,
    gammas=(0.0, 0.5, 1.0),
    n_rare=3,
    seed=0,
)
print(summarize_gamma_sweep(results))

In [ ]:
import matplotlib.pyplot as plt

# Plot last-n rare-class accuracy vs communication round for each method.
plot_order = [
    ('fedavg',        'FedAvg'),
    ('static',        'Static Reparam.'),
    ('fedgraa_g0.0',  'FedGRAA (g=0)'),
    ('fedgraa_g0.5',  'FedGRAA (g=0.5)'),
    ('fedgraa_g1.0',  'FedGRAA (g=1)'),
]

plt.figure(figsize=(8, 5))
for key, label in plot_order:
    if key in results:
        h = results[key].history
        plt.plot(h.rounds, [a * 100 for a in h.last_n_acc], marker='o', label=label)
plt.xlabel('Communication round')
plt.ylabel('Last-n rare-class accuracy (%)')
plt.title('CIC-IDS-2017 — rare-class accuracy (Figure 1 style)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5 — Optional extras

* **n-sweep** (Figures 2-3): FedGRAA robustness as the number of protected rare clients grows, n ∈ {3, 5, 7}.
* **UNSW-NB15**: re-run the whole thing on the second dataset by pointing `--data-dir` at its CSVs.

In [ ]:
# Figures 2-3: FedGRAA(g=1) vs baselines for n in {3, 5, 7}.
!python -m fedgraa n-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --gamma 1 --rounds 200

In [ ]:
# Optional: the same gamma sweep on UNSW-NB15.
# Set UNSW_DATA_DIR to the folder holding the UNSW-NB15 CSVs, then run.
# UNSW_DATA_DIR = '/content/drive/MyDrive/datasets/UNSW-NB15'
# !python -m fedgraa gamma-sweep --dataset UNSW-NB15 --data-dir "{UNSW_DATA_DIR}" --rounds 200